# KND_04 - Step 2: Create the answer we want to predict

## What are we doing?

We are adding a column called **long_stay_30**:

- **0:** a completed stay of 30 days or less.
- **1:** a stay of more than 30 days.

This is our **target**: the answer the model will learn to predict. The 30-day limit is our project planning choice, not an official shelter rule.

Examples (made up to explain the rule):

- An animal leaves after 12 days: **0**.
- An animal leaves after exactly 30 days: **0**.
- An animal leaves after 45 days: **1**.
- An animal is still there after 10 days: **we do not know yet**.
- An animal is still there after 45 days: **1**, if the blank outcome fields really mean the stay is ongoing.

The target covers the time until **any recorded outcome**, not just adoption. It does not tell us the reason an animal left or its exact kennel occupancy.

## Work as a group

- **Member 1:** run the notebook and explain the target rules.
- **Member 2:** review the counts and reasons for setting records aside.
- **Member 3:** check that missing birth dates are not removed here; age handling comes later.
- **Member 4:** rerun everything and confirm the checks pass.

Everyone should be able to explain the 10-day and 45-day examples.

## Before running

Use the same original CSV as Notebook 1. Place it beside this notebook, or upload both files to Colab. Run the cells in order. This notebook works on its own; Notebook 1 does not have to remain open.

Source: [County of Sonoma, Animal Shelter Intake and Outcome](https://data.sonomacounty.ca.gov/Government/Animal-Shelter-Intake-and-Outcome/924a-vesw), California, USA. Our saved extract was downloaded on 15 September 2026.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd

filename = "Sonoma_Animal_Shelter_2023-2026_RAW_9940.csv"
candidates = [Path(filename), Path("outputs") / filename, Path("/content") / filename]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the original CSV beside this notebook, or upload it to Colab.")

EXPECTED_HASH = "fd465b0dc5178bd13d0b9e70959320137f9526e635fe982c5ffc2c115faf61e0"
raw_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert raw_hash == EXPECTED_HASH, "This is a different file. Check the dataset version before continuing."

raw = pd.read_csv(DATA_PATH, dtype={"id": "string", "impound_number": "string"})
df = raw.copy()
df["source_row_number"] = range(1, len(df) + 1)
print("Loaded", len(df), "records. The original CSV will not be changed.")

Loaded 9940 records. The original CSV will not be changed.


## 1. Fix the observation date

Our saved data shows the situation up to **14 September 2026**. We must use that date, even if we run this notebook later.

For example, running this notebook in October does not give us new information about an animal last observed in September.

We read the dates and calculate:

- **stay_days:** outcome date minus arrival date, for completed stays.
- **observed_days:** observation date minus arrival date.

These are label-checking columns. **They must not become prediction inputs.**

In [2]:
OBSERVATION_DATE = pd.Timestamp("2026-09-14")
THRESHOLD_DAYS = 30

intake = pd.to_datetime(df["intake_date"], errors="coerce").dt.normalize()
outcome = pd.to_datetime(df["outcome_date"], errors="coerce").dt.normalize()
has_outcome_date = df["outcome_date"].fillna("").astype(str).str.strip().ne("")
has_outcome_type = df["outcome_type"].fillna("").astype(str).str.strip().ne("")

df["stay_days"] = (outcome - intake).dt.days
df["observed_days"] = (OBSERVATION_DATE - intake).dt.days
df["exclusion_reason"] = ""
print("Observation date:", OBSERVATION_DATE.date())

Observation date: 2026-09-14


## 2. Set aside records that need checking

We give each excluded row **one reason**, using the order below. That makes the counts easy to add up.

**Same animal and same arrival date:** We set aside the whole group for now because it may contain repeated or conflicting entries. We are not claiming all these rows are errors. Keep them for manual review; a confirmed legitimate admission may be restored later.

An animal returning in a different month is not removed by this rule.

We also check unreadable dates, outcomes before arrival, outcomes after our observation date, and records with only one of the two outcome fields filled in.

In [3]:
# Assign only the first applicable reason, so excluded rows are not counted twice.
def set_aside(mask, reason):
    df.loc[mask & df["exclusion_reason"].eq(""), "exclusion_reason"] = reason

same_day_visit = pd.DataFrame({"id": df["id"], "intake": intake}).duplicated(
    subset=["id", "intake"], keep=False
) & df["id"].notna() & intake.notna()
set_aside(same_day_visit, "Same animal and intake date: review needed")
set_aside(intake.isna() | (intake > OBSERVATION_DATE), "Missing, unreadable or future intake date")
set_aside(has_outcome_date & outcome.isna(), "Unreadable outcome date")
set_aside(outcome < intake, "Outcome before intake")
set_aside(outcome > OBSERVATION_DATE, "Outcome after observation date")
set_aside(has_outcome_date != has_outcome_type, "Only one outcome field is recorded")

print(df.loc[df["exclusion_reason"].ne(""), "exclusion_reason"].value_counts().to_string())

exclusion_reason
Same animal and intake date: review needed    24
Outcome after observation date                 1
Only one outcome field is recorded             1


## 3. Create the target

For a valid completed stay, compare the calculated stay with 30 days.

For an ongoing stay:

- More than 30 days already observed: label **1**.
- 30 days or less observed: set aside, because the answer is still unknown.

**Assumption:** both outcome fields being blank means the animal is still recorded as staying. Missing outcomes could also be delayed or incomplete records. We mark these assumed-ongoing labels separately, and later compare results using completed stays only.

In [4]:
valid = df["exclusion_reason"].eq("")
completed = valid & has_outcome_date & has_outcome_type
ongoing = valid & ~has_outcome_date & ~has_outcome_type

df["long_stay_30"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
df["label_source"] = ""

df.loc[completed, "long_stay_30"] = (df.loc[completed, "stay_days"] > THRESHOLD_DAYS).astype(int)
df.loc[completed, "label_source"] = "Recorded completed stay"

ongoing_long = ongoing & (df["observed_days"] > THRESHOLD_DAYS)
df.loc[ongoing_long, "long_stay_30"] = 1
df.loc[ongoing_long, "label_source"] = "Assumed ongoing: already over 30 days"

set_aside(ongoing & (df["observed_days"] <= THRESHOLD_DAYS), "Ongoing stay: answer not known yet")
known_label_count = int(df["long_stay_30"].notna().sum())
print("Records with a known or assumed-ongoing label:", known_label_count)
print("Labels based on assumed ongoing stays:", int(ongoing_long.sum()))

Records with a known or assumed-ongoing label: 9803
Labels based on assumed ongoing stays: 100


## 4. Give recent arrivals enough follow-up time

Imagine we include a dog that arrived five days ago and left after two days. Its short-stay answer is known. But a dog that arrived five days ago and is still there has no answer yet. Keeping only the recent animals that have already left would favour short stays.

To avoid this, our main dataset includes arrivals that had **at least 31 days of possible follow-up** by the observation date. With our dates, that means arrival on or before **14 August 2026**.

This does not mean every included animal actually stayed 31 days. An animal that arrived in 2025 and left after two days can still be included.

This is our chosen follow-up rule, not a claim that recent completed records are incorrect.

In [5]:
LATEST_INTAKE_DATE = OBSERVATION_DATE - pd.Timedelta(days=THRESHOLD_DAYS + 1)
set_aside(intake > LATEST_INTAKE_DATE, "Recent arrival: less than 31 days of possible follow-up")

labelled = df.loc[df["exclusion_reason"].eq("")].copy()
excluded = df.loc[df["exclusion_reason"].ne("")].copy()
labelled["long_stay_30"] = labelled["long_stay_30"].astype(int)

print("Latest included arrival:", LATEST_INTAKE_DATE.date())
print("Retained records:", len(labelled))
print("Set aside:", len(excluded))
print("\nReasons for setting records aside:")
print(excluded["exclusion_reason"].value_counts().to_string())

Latest included arrival: 2026-08-14
Retained records: 9670
Set aside: 270

Reasons for setting records aside:
exclusion_reason
Recent arrival: less than 31 days of possible follow-up    133
Ongoing stay: answer not known yet                         111
Same animal and intake date: review needed                  24
Outcome after observation date                               1
Only one outcome field is recorded                           1


## 5. Check the result

For our fixed CSV and these rules, we expect **9,670 retained records**:

- **7,040** stays of 30 days or less.
- **2,630** stays of more than 30 days.

These checks help catch an accidental change. Do not change the rules just to force the expected totals; investigate any mismatch.

This is the dataset after target and eligibility checks. **Missing-age handling, feature preparation and data splitting are still unfinished.** Do not call this a fully cleaned or training-ready file.

In [6]:
assert len(raw) == len(labelled) + len(excluded)
assert labelled["source_row_number"].is_unique
assert set(labelled["source_row_number"]).isdisjoint(excluded["source_row_number"])
assert labelled["long_stay_30"].notna().all()
assert (labelled["observed_days"] >= 31).all()

counts = labelled["long_stay_30"].value_counts().sort_index()
assert known_label_count == 9803
assert len(labelled) == 9670
assert counts.to_dict() == {0: 7040, 1: 2630}

print("30 days or less:", counts[0])
print("More than 30 days:", counts[1])
print("Long-stay percentage:", round(100 * counts[1] / len(labelled), 2))
print("\nAll row-count and label checks passed.")

30 days or less: 7040
More than 30 days: 2630
Long-stay percentage: 27.2

All row-count and label checks passed.


## 6. Save the results

The next cell creates a **step2_results** folder beside the original CSV:

- **labelled_records.csv:** retained records with labels; not yet a model-input file.
- **excluded_records_for_review.csv:** all set-aside records and their reasons.
- **exclusion_counts.csv:** the number set aside for each reason.
- **target_notes.json:** source fingerprint, dates, assumptions and counts.

Running this cell again replaces these generated files. It never replaces the original CSV. Keep these working data files in your group workspace.

**Important:** outcome columns, `days_in_shelter`, `stay_days`, `observed_days`, `label_source`, `exclusion_reason` and row numbers are not model inputs. Identifiers are useful for record checks and group-aware splitting, not for prediction. Later we will explicitly select arrival-time inputs rather than passing this whole file into a model.

In [7]:
RESULTS_DIR = DATA_PATH.resolve().parent / "step2_results"
RESULTS_DIR.mkdir(exist_ok=True)
labelled.to_csv(RESULTS_DIR / "labelled_records.csv", index=False)
excluded.to_csv(RESULTS_DIR / "excluded_records_for_review.csv", index=False)
reason_counts = excluded["exclusion_reason"].value_counts().rename_axis("reason").reset_index(name="records")
reason_counts.to_csv(RESULTS_DIR / "exclusion_counts.csv", index=False)

notes = {
    "source_url": "https://data.sonomacounty.ca.gov/Government/Animal-Shelter-Intake-and-Outcome/924a-vesw",
    "source_file": filename,
    "source_sha256": raw_hash,
    "observation_date": str(OBSERVATION_DATE.date()),
    "latest_included_intake": str(LATEST_INTAKE_DATE.date()),
    "target": "long_stay_30: 1 if stay exceeds 30 days, otherwise 0",
    "raw_rows": len(raw), "retained_rows": len(labelled), "excluded_rows": len(excluded),
    "short_stays": int(counts[0]), "long_stays": int(counts[1]),
    "assumed_ongoing_labels": int(labelled["label_source"].str.startswith("Assumed ongoing").sum()),
    "limitations": [
        "Both outcome fields blank is treated as ongoing; delayed updates could affect labels.",
        "Same-animal, same-date groups are set aside for review, not proven errors.",
        "Duration covers any outcome, not adoption alone or exact kennel occupancy.",
        "Predictor cleaning, time/group splitting and training-only transformations remain unfinished."
    ]
}
(RESULTS_DIR / "target_notes.json").write_text(json.dumps(notes, indent=2), encoding="utf-8")
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == raw_hash
print("Saved four files in:", RESULTS_DIR)
print("Original CSV unchanged.")

Saved four files in: C:\Users\ishin\Documents\Codex\2026-09-15\we-x20\outputs\step2_results
Original CSV unchanged.


## Your group task before moving on

Run every cell and confirm that the checks pass. Then answer these questions in your own words:

1. Why does exactly 30 days get label 0?
2. Why is an ongoing 10-day stay not labelled 0?
3. What assumption lets us label an ongoing 45-day stay as 1?
4. Why do we set aside recent arrivals even when some already have outcomes?
5. Why are same-animal, same-date rows held for review rather than called confirmed duplicates?
6. Why is `days_in_shelter` useful for understanding the data but unsuitable as an input?

**Record each member's actual work:** name, task completed, finding, and explanation. Running a supplied notebook should be recorded honestly as running, checking and explaining it; note any changes your group makes.

### Simple viva explanation

"We created a target to show whether an animal stayed for more than 30 days. We set aside records where the answer was unclear and used a fixed observation date. We also excluded recent arrivals so that every included arrival had enough possible follow-up time. We kept 9,670 records at this stage, and saved the excluded records with their reasons."

### Next step

Agree on a time-based training, validation and test split, including checks for repeated animals and enough time for labels to become known. Then explore the training data in detail and choose preprocessing rules. This keeps future test data from influencing our choices.